In [1]:
# Core data processing and scientific computing
import pandas as pd
import numpy as np
import ast

# Machine learning and feature extraction
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# Collections for data structures
from collections import defaultdict


In [2]:
print("Aligning datasets to ensure common games...")

# Load both datasets
games_processed_all = pd.read_pickle("games_processed.pkl")
recommendations_df = pd.read_pickle("recommendations_processed.pkl")

# Get common games
common_game_ids = set(games_processed_all['app_id']).intersection(set(recommendations_df['app_id']))
print(f"Found {len(common_game_ids)} common games between datasets")

# Filter both datasets
games_processed_all = games_processed_all[games_processed_all['app_id'].isin(common_game_ids)]
recommendations_df = recommendations_df[recommendations_df['app_id'].isin(common_game_ids)]

print(f"Filtered games_processed_all shape: {games_processed_all.shape}")
print(f"Filtered recommendations_df shape: {recommendations_df.shape}")

Aligning datasets to ensure common games...
Found 2850 common games between datasets
Filtered games_processed_all shape: (2850, 7)
Filtered recommendations_df shape: (1034570, 8)


In [3]:
# Modify the existing TF-IDF cell
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from tqdm.notebook import tqdm

print("Loading and processing games data...")
games_processed_all = pd.read_pickle("games_processed.pkl")

# Print sample of tags to understand the format
print("\nSample of tags before processing:")
print(games_processed_all["tags"].head())

# ✅ Step 1: Process tags
print("\nProcessing tags...")
# If tags are already space-separated strings, we can use them directly
games_processed_all["tags_text"] = games_processed_all["tags"].apply(
    lambda x: x if isinstance(x, str) else " ".join(x) if isinstance(x, list) else ""
)

# ✅ Step 2: Filter out entries with very short or empty tags
print("Filtering entries...")
games_processed_all = games_processed_all[games_processed_all["tags_text"].str.strip() != ""]

# ✅ Step 3: TF-IDF vectorization for tags
print("Performing TF-IDF vectorization for tags...")
vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
tfidf_matrix = vectorizer.fit_transform(games_processed_all["tags_text"])

# ✅ Step 4: Process ratings
print("Processing ratings...")
# Normalize ratings to 0-1 scale
scaler = StandardScaler()
ratings_normalized = scaler.fit_transform(games_processed_all[["rating"]].fillna(0))

# ✅ Step 5: Combine TF-IDF and ratings
print("Combining TF-IDF and ratings...")
tfidf_array = tfidf_matrix.toarray()
combined_features = np.hstack([tfidf_array, ratings_normalized])

# ✅ Step 6: Create DataFrame with combined features
print("Creating combined features DataFrame...")
feature_names = list(vectorizer.get_feature_names_out()) + ['rating']
tfidf_df = pd.DataFrame(combined_features, 
                       index=games_processed_all["app_id"], 
                       columns=feature_names)

print("✅ tfidf_df successfully created with shape:", tfidf_df.shape)

# Print sample of processed data
print("\nSample of processed data:")
print("\nTags text sample:")
print(games_processed_all["tags_text"].head())
print("\nFeature names sample:")
print(feature_names[:10])

Loading and processing games data...

Sample of tags before processing:
0    Simulation Tower Defense Strategy Turn-Based S...
1                                  Simulation Strategy
2    Game Development Animation & Modeling Design &...
3    Management City Builder Base Building Colony S...
4    Point & Click Comedy 2D Noir Cute Detective Fu...
Name: tags, dtype: string

Processing tags...
Filtering entries...
Performing TF-IDF vectorization for tags...
Processing ratings...
Combining TF-IDF and ratings...
Creating combined features DataFrame...
✅ tfidf_df successfully created with shape: (4146, 478)

Sample of processed data:

Tags text sample:
0    Simulation Tower Defense Strategy Turn-Based S...
1                                  Simulation Strategy
2    Game Development Animation & Modeling Design &...
3    Management City Builder Base Building Colony S...
4    Point & Click Comedy 2D Noir Cute Detective Fu...
Name: tags_text, dtype: object

Feature names sample:
['1980s', '1990',

In [4]:
def create_cold_start_split(recommendations_df, test_size=0.15, val_size=0.15,
                            timestamp_col='date', cold_start_user_frac=0.01, seed=42):
    import numpy as np
    np.random.seed(seed)

    # Sort by timestamp if available
    if timestamp_col in recommendations_df.columns:
        recommendations_df = recommendations_df.sort_values(timestamp_col)

    # Identify cold-start users and items
    all_users = recommendations_df['user_id'].unique()
    all_items = recommendations_df['app_id'].unique()

    n_cold_users = int(len(all_users) * cold_start_user_frac)

    cold_users = np.random.choice(all_users, size=n_cold_users, replace=False)

    # Cold-start test data
    cold_user_df = recommendations_df[recommendations_df['user_id'].isin(cold_users)]

    # Remove cold users/items from the remaining dataset
    warm_df = recommendations_df[
        ~recommendations_df['user_id'].isin(cold_users)
    ]

    # Now split warm_df by user chronologically
    user_groups = warm_df.groupby('user_id')

    train_data = []
    val_data = []
    test_data = []

    for user_id, user_data in user_groups:
        n = len(user_data)
        if n < 3:
            continue

        n_test = max(1, int(n * test_size))
        n_val = max(1, int(n * val_size))

        user_train = user_data.iloc[:-n_test-n_val]
        user_val = user_data.iloc[-n_test-n_val:-n_test]
        user_test = user_data.iloc[-n_test:]

        train_data.append(user_train)
        val_data.append(user_val)
        test_data.append(user_test)

    train_df = pd.concat(train_data)
    val_df = pd.concat(val_data)
    test_df = pd.concat(test_data)

    return train_df, val_df, test_df, cold_user_df


train_df, val_df, test_df, cold_user_df = create_cold_start_split(recommendations_df)


In [5]:
# Modify the user profile creation cell
print("Building user profiles with combined features...")
user_profiles_tfidf = {}

for user_id in tqdm(train_df["user_id"].unique(), desc="Creating user profiles"):
    # Get user's games
    user_games = train_df[train_df["user_id"] == user_id]
    
    # Get feature vectors for user's games
    app_ids = user_games["app_id"]
    valid_ids = app_ids[app_ids.isin(tfidf_df.index)]
    
    if not valid_ids.empty:
        # Get feature vectors
        vectors = tfidf_df.loc[valid_ids]
        
        # Get user's ratings for these games
        user_ratings = user_games[user_games["app_id"].isin(valid_ids)]["rating"].values
        
        # Weight the features by user's ratings
        if len(user_ratings) > 0:
            # Normalize ratings to 0-1 scale if not already
            user_ratings = (user_ratings - user_ratings.min()) / (user_ratings.max() - user_ratings.min() + 1e-6)
            # Apply rating weights
            weighted_vectors = vectors.multiply(user_ratings, axis=0)
            # Calculate mean profile
            user_profiles_tfidf[user_id] = weighted_vectors.mean(axis=0).values.reshape(1, -1)
        else:
            # If no ratings, use simple mean
            user_profiles_tfidf[user_id] = vectors.mean(axis=0).values.reshape(1, -1)

print(f"✅ Built combined feature profiles for {len(user_profiles_tfidf)} users.")

Building user profiles with combined features...


Creating user profiles:   0%|          | 0/34602 [00:00<?, ?it/s]

✅ Built combined feature profiles for 34602 users.


In [6]:

# ✅ recommend_tfidf_contentwith fallback for cold start users

def recommend_tfidf_content(user_id, top_n=10):
    if user_id not in user_profiles_tfidf:
        fallback = games_processed_all.sort_values(by="user_reviews", ascending=False).head(top_n)
        recs = fallback[["app_id", "title"]].copy()
        recs.insert(0, "user_id", user_id)
        return recs

    user_vector = user_profiles_tfidf[user_id]
    similarities = cosine_similarity(user_vector, tfidf_df.values).flatten()

    played = set(train_df[train_df["user_id"] == user_id]["app_id"])
    sorted_indices = similarities.argsort()[::-1]
    recommended_ids = [tfidf_df.index[i] for i in sorted_indices if tfidf_df.index[i] not in played][:top_n]

    recs = games_processed_all[games_processed_all["app_id"].isin(recommended_ids)][["app_id", "title"]].copy()
    recs.insert(0, "user_id", user_id)
    return recs

print("✅ Recommender supports fallback for cold users and can be evaluated.")


✅ Recommender supports fallback for cold users and can be evaluated.


In [7]:

# 🔍 Demo: View a TF-IDF user profile and their top 5 content-based recommendations

# Pick a sample user with a TF-IDF profile
#sample_user_id = next(iter(user_profiles_tfidf.keys()))
# 🔍 Demo: Recommend games for any user (TF-IDF or fallback)

sample_user_id = 1239  # Change this to test any user

if sample_user_id in user_profiles_tfidf:
    # ✅ TF-IDF path
    user_vector = user_profiles_tfidf[sample_user_id]
    similarities = cosine_similarity(user_vector, tfidf_df.values).flatten()

    played = set(train_df[train_df["user_id"] == sample_user_id]["app_id"])
    sorted_indices = similarities.argsort()[::-1]
    recommended_ids = [tfidf_df.index[i] for i in sorted_indices if tfidf_df.index[i] not in played][:5]

    recommended_games = games_processed_all[games_processed_all["app_id"].isin(recommended_ids)][["app_id", "title"]].reset_index(drop=True)

    # Show user's top tags
    user_tag_scores = pd.Series(user_vector.flatten(), index=tfidf_df.columns).sort_values(ascending=False)
    top_user_tags = user_tag_scores.head(10)

    print(f"🧑‍💻 TF-IDF Recommendation for User ID: {sample_user_id}")
    print("\n🔝 Top Tags for this User:")
    display(top_user_tags)

    print("\n🎮 Top 5 Game Recommendations:")
    display(recommended_games)

else:
    # 🧊 Fallback for cold start user
    recommendations = recommend_tfidf_content(sample_user_id, top_n=5)
    print(f"🧊 Fallback Recommendation for Cold Start User ID: {sample_user_id}")
    display(recommendations)


🧑‍💻 TF-IDF Recommendation for User ID: 1239

🔝 Top Tags for this User:


rating      0.302526
op          0.033698
co          0.033698
survival    0.029903
mod         0.029562
craft       0.026488
value       0.021745
replay      0.021745
to          0.020060
play        0.020060
dtype: float64


🎮 Top 5 Game Recommendations:


,app_id,title
0,1117850,Cuphead - The Delicious Last Course
1,597170,Clone Drone in the Danger Zone
2,1123450,Chicory: A Colorful Tale
3,1536610,OpenTTD
4,1255980,Portal Reloaded


In [8]:
# ✅ Autoencoder-based Content-Based Recommender (Tag Embedding)

from sklearn.neural_network import MLPRegressor
from sklearn.metrics.pairwise import cosine_similarity

# Step 1: Use tag matrix from already encoded tags
# Step 1: Use TF-IDF matrix instead of one-hot encoding
tag_matrix = tfidf_df.values
app_ids = tfidf_df.index.values

# Step 2: Train a shallow autoencoder
autoencoder = MLPRegressor(hidden_layer_sizes=(50,), max_iter=2000, random_state=42)
autoencoder.fit(tag_matrix, tag_matrix)

# Step 3: Get latent representations (encoded tag vectors)
encoded_vectors = autoencoder.predict(tag_matrix)
latent_df = pd.DataFrame(encoded_vectors, index=app_ids)


# Step 4: Build user profiles in latent space
user_profiles_autoenc = {}
for user_id in train_df["user_id"].unique():
    app_ids_user = train_df[train_df["user_id"] == user_id]["app_id"]
    valid_ids = app_ids_user[app_ids_user.isin(latent_df.index)]
    user_vector = latent_df.loc[valid_ids].mean(axis=0)
    if not user_vector.isna().any():
        user_profiles_autoenc[user_id] = user_vector.values.reshape(1, -1)

# Step 5: Recommendation function

def recommend_autoencoder_content(user_id, top_n=10):
    if user_id not in user_profiles_autoenc:
        fallback = games_processed_all.sort_values(by="user_reviews", ascending=False).head(top_n)
        recs = fallback[["app_id", "title"]].copy()
        recs.insert(0, "user_id", user_id)
        return recs

    user_vector = user_profiles_autoenc[user_id]
    similarities = cosine_similarity(user_vector, latent_df.values).flatten()

    played = set(train_df[train_df["user_id"] == user_id]["app_id"])
    sorted_indices = similarities.argsort()[::-1]
    recommended_ids = [latent_df.index[i] for i in sorted_indices if latent_df.index[i] not in played][:top_n]

    recs = games_processed_all[games_processed_all["app_id"].isin(recommended_ids)][["app_id", "title"]].copy()
    recs.insert(0, "user_id", user_id)
    return recs



In [9]:
# 🔍 Demo: Recommend games for any user using Autoencoder (with fallback)

sample_user_id = 1239  # Change this to test another user

recommendations = recommend_autoencoder_content(sample_user_id, top_n=5)

if sample_user_id in user_profiles_autoenc:
    print(f"\U0001f9e0 Autoencoder-Based Recommendations for User ID: {sample_user_id}")
        
    # Show top 10 latent dimensions for the user
    user_vector = user_profiles_autoenc[sample_user_id]
    user_latent_scores = pd.Series(user_vector.flatten(), index=[f"latent_{i}" for i in range(user_vector.shape[1])])
    top_latent_features = user_latent_scores.sort_values(ascending=False).head(10)

    print("\n🔝 Top Latent Dimensions for this User (Autoencoder):")
    display(top_latent_features)
else:
    print(f"\U0001f9ca Fallback for Cold Start User ID: {sample_user_id}")

display(recommendations)

🧠 Autoencoder-Based Recommendations for User ID: 1239

🔝 Top Latent Dimensions for this User (Autoencoder):


latent_477    1.179835
latent_344    0.076605
latent_382    0.075716
latent_3      0.074856
latent_70     0.071992
latent_180    0.070784
latent_296    0.070267
latent_216    0.070210
latent_406    0.069498
latent_82     0.060952
dtype: float64

,user_id,app_id,title
63,1239,2085360,Insomnia: Theater in the Head
357,1239,1274140,One Dreamer: Prologue
2348,1239,1905180,OBS Studio
2909,1239,1684660,natsuno-kanata - beyond the summer
3836,1239,1545450,Incredibox


In [10]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from collections import defaultdict
import numpy as np

def evaluate_recommender(model_function, df_test, k=5):
    """
    Evaluate a content-based recommender system using precision, recall, f1, accuracy, precision@k, ndcg@k.
    
    Parameters:
    - model_function: recommendation function taking user_id and top_n=k
    - df_test: test set with user_id and app_id columns
    - k: number of recommendations per user

    Returns:
    - metrics: dictionary with average scores
    """
    true_positives = 0
    total_recommended = 0
    total_relevant = 0

    precision_scores = []
    recall_scores = []
    f1_scores = []
    accuracy_scores = []
    ndcg_scores = []

    users_evaluated = 0

    grouped_test = df_test.groupby("user_id")
    
    for user_id, group in grouped_test:
        true_items = set(group["app_id"])
        if not true_items:
            continue

        recs = model_function(user_id, top_n=k)
        predicted_items = list(recs["app_id"].dropna())
        
        if not predicted_items:
            continue

        y_true = [1 if app_id in true_items else 0 for app_id in predicted_items]
        y_pred = [1] * len(predicted_items)  # recommender always predicts relevance

        # Basic metrics
        precision_scores.append(precision_score(y_true, y_pred, zero_division=0))
        recall_scores.append(recall_score(y_true, y_pred, zero_division=0))
        f1_scores.append(f1_score(y_true, y_pred, zero_division=0))
        accuracy_scores.append(accuracy_score(y_true, y_pred))

        # Precision@k
        hits = sum(y_true)
        precision_at_k = hits / k
        precision_scores.append(precision_at_k)

        # NDCG@k
        dcg = sum([int(relevant) / np.log2(idx + 2) for idx, relevant in enumerate(y_true)])
        idcg = sum([1.0 / np.log2(i + 2) for i in range(min(len(true_items), k))])
        ndcg = dcg / idcg if idcg > 0 else 0.0
        ndcg_scores.append(ndcg)

        users_evaluated += 1

    # Aggregate scores
    metrics = {
        "Users Evaluated": users_evaluated,
        "Avg Precision": round(np.mean(precision_scores), 4),
        "Avg Recall": round(np.mean(recall_scores), 4),
        "Avg F1": round(np.mean(f1_scores), 4),
        "Avg Accuracy": round(np.mean(accuracy_scores), 4),
        "Avg NDCG@k": round(np.mean(ndcg_scores), 4)
    }

    return metrics

# Evaluate on the regular test set
results_normal = evaluate_recommender(recommend_autoencoder_content, test_df, k=5)
print(f"\n✅ Evaluation Results (Normal Users): {len(test_df)} Recommendations")
for metric, value in results_normal.items():
    print(f"{metric}: {value}")

# Evaluate on the cold-start user test set
results_cold = evaluate_recommender(recommend_autoencoder_content, cold_user_df, k=5)
print(f"\n🧊 Evaluation Results (Cold-Start Users): {len(cold_user_df)} Users")
for metric, value in results_cold.items():
    print(f"{metric}: {value}")


✅ Evaluation Results (Normal Users): 35794 Recommendations
Users Evaluated: 34602
Avg Precision: 0.006
Avg Recall: 0.0301
Avg F1: 0.01
Avg Accuracy: 0.006
Avg NDCG@k: 0.017

🧊 Evaluation Results (Cold-Start Users): 10446 Users
Users Evaluated: 8390
Avg Precision: 0.0322
Avg Recall: 0.1566
Avg F1: 0.0533
Avg Accuracy: 0.0322
Avg NDCG@k: 0.0887


In [11]:
# Evaluate your TF-IDF recommender
# Evaluate on the regular test set
results_normal = evaluate_recommender(recommend_tfidf_content, test_df, k=5)
print(f"\n✅ Evaluation Results (Normal Users): {len(test_df)} Recommendations")
for metric, value in results_normal.items():
    print(f"{metric}: {value}")

# Evaluate on the cold-start user test set
results_cold = evaluate_recommender(recommend_tfidf_content, cold_user_df, k=5)
print(f"\n🧊 Evaluation Results (Cold-Start Users): {len(cold_user_df)} Users")
for metric, value in results_cold.items():
    print(f"{metric}: {value}")


✅ Evaluation Results (Normal Users): 35794 Recommendations
Users Evaluated: 34602
Avg Precision: 0.0028
Avg Recall: 0.0142
Avg F1: 0.0047
Avg Accuracy: 0.0028
Avg NDCG@k: 0.0077

🧊 Evaluation Results (Cold-Start Users): 10446 Users
Users Evaluated: 8390
Avg Precision: 0.0322
Avg Recall: 0.1566
Avg F1: 0.0533
Avg Accuracy: 0.0322
Avg NDCG@k: 0.0887
